# FaceProof 08 — extensions C2 + C3 (full: SD **and** T2I generators)

**Day 8.** Completes all four detector conditions and evaluates the two *trained* detectors on
every unseen generator (SD + SDXL/Flux/DALL-E), so the frozen-vs-fine-tuned generalization
question is fully answered.
- **C2** — ResNet-50 fine-tuned end-to-end.
- **C3** — DCT log-spectrum + linear SVM (frequency baseline).

Self-contained: clones the repo, rebuilds splits if needed, and (re)downloads SFHQ-T2I only if
absent. **GPU on.**

## 1. Setup

In [ ]:
REPO_URL = "https://github.com/Ezed9/faceProof.git"
BRANCH   = "main"
!git clone -b $BRANCH $REPO_URL
%cd faceProof
!pip install -q torch torchvision scikit-learn scipy joblib pyyaml kaggle
import sys, os
sys.path.append(os.getcwd())

In [ ]:
from google.colab import drive
from pathlib import Path
import numpy as np, pandas as pd
drive.mount('/content/drive')
ROOT          = Path('/content/drive/MyDrive/faceproof')
CROPS_ROOT    = ROOT / 'data' / 'crops'
PROBES_ROOT   = ROOT / 'models' / 'probes'
REPORTS_ROOT  = ROOT / 'reports'
MANIFEST      = ROOT / 'data' / 'manifest.csv'
RESULTS_CSV   = REPORTS_ROOT / 'results.csv'
(REPORTS_ROOT/'figures').mkdir(parents=True, exist_ok=True); PROBES_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
import csv
RESULT_FIELDS = ['condition','train_gen','test_gen','corruption','strength','seed','metric','value']
def log_result(**row):
    full = {k: row.get(k, '') for k in RESULT_FIELDS}
    new = not RESULTS_CSV.exists()
    with open(RESULTS_CSV, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=RESULT_FIELDS)
        if new: w.writeheader()
        w.writerow(full)

In [ ]:
import yaml, torch
from src.data import build_manifest, make_splits
from src.metrics import auroc, eer

df = pd.read_csv(MANIFEST)
if 'split' not in df.columns:                      # self-contained: rebuild splits if missing
    sources = {'real':{'dir':str(CROPS_ROOT/'real'),'label':0,'generator':'real'},
               'stylegan':{'dir':str(CROPS_ROOT/'stylegan'),'label':1,'generator':'stylegan'},
               'sd':{'dir':str(CROPS_ROOT/'sd'),'label':1,'generator':'stable_diffusion'}}
    df = make_splits(build_manifest(sources), train_generators=['stylegan'],
                     test_unseen=['stable_diffusion'], seed=13)
dcfg = yaml.safe_load(open('configs/data.yaml')); IMG, Q = dcfg['image_size'], dcfg['jpeg_match_quality']
fcfg = yaml.safe_load(open('configs/model.yaml'))['finetune_resnet']
DEV  = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device', DEV)

# Path-list evaluation groups (each has BOTH classes -> AUROC well-defined).
real_test = df[(df['label']==0) & (df['split']=='test_indist')]
def paths_labels(sub):
    return sub['path'].tolist(), sub['label'].values
EVAL = {
    'stylegan_indist': paths_labels(df[df['split']=='test_indist']),
    'stable_diffusion': paths_labels(pd.concat([real_test, df[df['generator']=='stable_diffusion']])),
}
REAL_TEST_PATHS = real_test['path'].tolist()[:3000]      # reused for the T2I eval
print({k: len(v[0]) for k,v in EVAL.items()})

## 2. C2 — fine-tuned ResNet-50 (train on `train`, early-stop on `val`)

In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision.models import resnet50, ResNet50_Weights
from PIL import Image

NORM = T.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
tf   = T.Compose([T.Resize((224,224)), T.ToTensor(), NORM])

class CropDS(Dataset):
    def __init__(self, sub): self.p=sub['path'].tolist(); self.y=sub['label'].tolist()
    def __len__(self): return len(self.p)
    def __getitem__(self,i): return tf(Image.open(self.p[i]).convert('RGB')), float(self.y[i])

def loader(sub, shuffle=False): return DataLoader(CropDS(sub), batch_size=fcfg['batch_size'], shuffle=shuffle, num_workers=2)

@torch.no_grad()
def c2_scores(paths, bs=64):
    c2.eval(); out=[]
    for i in range(0, len(paths), bs):
        xb = torch.stack([tf(Image.open(p).convert('RGB')) for p in paths[i:i+bs]]).to(DEV)
        out.append(torch.sigmoid(c2(xb).squeeze(1)).cpu().numpy())
    return np.concatenate(out)

@torch.no_grad()
def val_auc():
    c2.eval(); yv,pv=[],[]
    for xb,yb in loader(df[df['split']=='val']):
        pv.append(torch.sigmoid(c2(xb.to(DEV)).squeeze(1)).cpu().numpy()); yv.append(yb.numpy())
    return auroc(np.concatenate(yv), np.concatenate(pv))

In [ ]:
c2 = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
c2.fc = nn.Linear(c2.fc.in_features, 1); c2 = c2.to(DEV)
opt = torch.optim.Adam(c2.parameters(), lr=fcfg['lr']); lossf = nn.BCEWithLogitsLoss()
best_auc, best_state, patience = -1.0, None, 0
for ep in range(fcfg['epochs']):
    c2.train()
    for xb,yb in loader(df[df['split']=='train'], shuffle=True):
        opt.zero_grad(); loss = lossf(c2(xb.to(DEV)).squeeze(1), yb.to(DEV)); loss.backward(); opt.step()
    a = val_auc(); print(f'epoch {ep}: val AUROC {a:.4f}')
    if a > best_auc: best_auc, best_state, patience = a, {k:v.cpu().clone() for k,v in c2.state_dict().items()}, 0
    else:
        patience += 1
        if patience >= fcfg['early_stopping_patience']: print('early stop'); break
c2.load_state_dict(best_state); c2.to(DEV)
torch.save(c2.state_dict(), PROBES_ROOT/'c2_resnet_finetune.pt')

### C2 evaluation: in-distribution + Stable Diffusion

In [ ]:
for tg,(paths,y) in EVAL.items():
    p = c2_scores(paths)
    log_result(condition='c2_resnet_finetune', train_gen='stylegan', test_gen=tg, corruption='none', seed=13, metric='AUROC', value=round(auroc(y,p),4))
    log_result(condition='c2_resnet_finetune', train_gen='stylegan', test_gen=tg, corruption='none', seed=13, metric='EER',   value=round(eer(y,p),4))
    print(f'C2 {tg}: AUROC {auroc(y,p):.4f}')

## 3. C3 — DCT log-spectrum + linear SVM

In [ ]:
from src.leakage_check import dct_log_features
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def dct_mat(paths):
    feats = [dct_log_features(p) for p in paths]
    keep  = np.array([f is not None for f in feats])
    return np.array([f for f in feats if f is not None]), keep

tr = df[df['split']=='train']
Xtr, k = dct_mat(tr['path'].tolist())
c3 = make_pipeline(StandardScaler(), LinearSVC(max_iter=5000)).fit(Xtr, tr['label'].values[k])
for tg,(paths,y) in EVAL.items():
    Xte, keep = dct_mat(paths); s = c3.decision_function(Xte)
    log_result(condition='c3_frequency', train_gen='stylegan', test_gen=tg, corruption='none', seed=13, metric='AUROC', value=round(auroc(y[keep],s),4))
    log_result(condition='c3_frequency', train_gen='stylegan', test_gen=tg, corruption='none', seed=13, metric='EER',   value=round(eer(y[keep],s),4))
    print(f'C3 {tg}: AUROC {auroc(y[keep],s):.4f}')

## 4. Decisive test: C2 + C3 on the far T2I generators (SDXL / Flux / DALL-E)

Frozen probes collapsed below chance here (Day 5). Do the *trained* detectors too?

In [ ]:
import glob
from google.colab import files
# Skip the (large) download if the SFHQ-T2I CSV is already present.
def find_t2i_csv():
    cs = [c for c in glob.glob('/content/**/*.csv', recursive=True) if 't2i' in c.lower() or 'sfhq' in c.lower()]
    return cs[0] if cs else None
if find_t2i_csv() is None:
    print('Upload kaggle.json:'); files.upload()
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !kaggle datasets download -d selfishgene/sfhq-t2i-synthetic-faces-from-text-2-image-models -p /content/t2i --unzip
print('T2I CSV:', find_t2i_csv())

In [ ]:
csv_path = find_t2i_csv(); assert csv_path, 'T2I CSV not found'
meta = pd.read_csv(csv_path)
COL_IMAGE, COL_GEN = 'image_filename', 'model_used'   # confirmed SFHQ-T2I schema
img_root = Path(csv_path).parent
index = {q.name: str(q) for q in img_root.rglob('*') if q.suffix.lower() in {'.jpg','.jpeg','.png'}}
meta['path'] = meta[COL_IMAGE].map(index.get)
meta = meta.dropna(subset=['path'])
meta = meta[meta[COL_GEN].isin(['SDXL','FLUX1_schnell','DALLE3'])].copy()   # 3 distinct families
assert len(meta), 'meta empty - check columns/download'
print(meta[COL_GEN].value_counts())

In [ ]:
import shutil
from src.preprocessing import preprocess_image

def crops_for(paths, tag):
    out = Path(f'/content/_t2i_{tag}')
    if out.exists(): shutil.rmtree(out)
    out.mkdir(parents=True)
    kept=[]
    for i,p in enumerate(paths):
        im = preprocess_image(p, IMG, Q, detector=None)     # same compression-matched pipeline
        if im is None: continue
        op = out/f'{i:06d}.jpg'; im.save(op,'JPEG',quality=Q); kept.append(str(op))
    return kept

rows=[]
for gen, grp in meta.groupby(COL_GEN):
    fakes = crops_for(grp['path'].tolist()[:3000], gen.replace('/','_'))
    paths = list(REAL_TEST_PATHS) + list(fakes)
    y     = np.r_[np.zeros(len(REAL_TEST_PATHS)), np.ones(len(fakes))]
    p2 = c2_scores(paths); au2 = auroc(y, p2)
    Xte, keep = dct_mat(paths); au3 = auroc(y[keep], c3.decision_function(Xte))
    log_result(condition='c2_resnet_finetune', train_gen='stylegan', test_gen=str(gen), corruption='none', seed=13, metric='AUROC', value=round(au2,4))
    log_result(condition='c3_frequency',       train_gen='stylegan', test_gen=str(gen), corruption='none', seed=13, metric='AUROC', value=round(au3,4))
    rows.append({'generator':gen,'C2_AUROC':round(au2,4),'C3_AUROC':round(au3,4),'C2_meanP_fake':round(p2[len(REAL_TEST_PATHS):].mean(),3)})
    print(f'{gen}: C2 {au2:.4f} | C3 {au3:.4f}')
print(pd.DataFrame(rows).to_string(index=False))
print('\n✅ Day 8 complete: C2 + C3 evaluated on SD AND on SDXL/Flux/DALL-E (results.csv)')